# Lab 27-05 · Inside a Laptop SoC

Companion to [27-05 · The PC SoC](../../chapters/27-05-pc-soc.md). Each part asks you to **predict** first, then run the cell. All parameters are the chapter's assumptions.

In [1]:
import math
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pcsoc as pc

## Part 1 · Deep idle and battery life

*Predict* the battery life at 85% deep-idle residency (8% active, 7% shallow).

In [2]:
soc = pc.residency_power_w([(0.08, 5.0), (0.07, 1.2), (0.85, 0.25)])
print(f'SoC {soc:.3f} W, life {pc.battery_hours(70, soc + 3.0):.1f} h')
deep = np.linspace(0.5, 0.9, 41)
life = [pc.battery_hours(70, pc.residency_power_w([(0.08, 5.0), (0.92 - d, 1.2), (d, 0.25)]) + 3.0) for d in deep]
fig, ax = plt.subplots(figsize=(6, 3.5)); ax.plot(100*deep, life)
ax.set_xlabel('deep-idle residency (%)'); ax.set_ylabel('battery life (h)'); ax.grid(True, alpha=0.3); fig.tight_layout()

SoC 0.697 W, life 18.9 h


## Part 2 · Turbo

*Predict* the turbo duration with τ = 56 s, from an idle average of 5 W.

In [3]:
print(f'tau 28 s: {pc.turbo_seconds(28, 64, 28, 5):.1f} s; tau 56 s: {pc.turbo_seconds(28, 64, 56, 5):.1f} s')
dt = 0.01; t = np.arange(0, 60, dt)
fig, ax = plt.subplots(figsize=(6, 3.5))
for tau in (28, 56):
    y, ys = 5.0, []
    a = 1 - math.exp(-dt / tau)
    for _ in t:
        p = 64.0 if y < 28.0 else 28.0
        y += a * (p - y); ys.append(y)
    ax.plot(t, ys, label=f'tau = {tau} s')
ax.axhline(28, color='k', lw=0.8); ax.set_xlabel('time (s)'); ax.set_ylabel('averaged power (W)'); ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

tau 28 s: 13.8 s; tau 56 s: 27.7 s


## Part 3 · Local models

*Predict* the largest 4-bit model that decodes at 10 tokens/s on 256 GB/s.

In [4]:
print(f'largest model: {256 / (10 * 0.5):.1f} billion parameters')
sizes = np.linspace(1, 120, 120)
fig, ax = plt.subplots(figsize=(6, 3.5))
for bwg in (136.5, 256):
    ax.semilogy(sizes, [pc.decode_tokens_s(bwg, s, 4) for s in sizes], label=f'{bwg} GB/s')
ax.axhline(10, color='k', lw=0.8); ax.set_xlabel('parameters (billions), 4-bit'); ax.set_ylabel('tokens/s (bound)'); ax.legend(); ax.grid(True, which='both', alpha=0.3); fig.tight_layout()

largest model: 51.2 billion parameters


## Part 4 · When do tiles pay?

*Predict* the smallest total area at which two equal 3 nm-class tiles in a $10 package (97% assembly yield) beat one die.

In [5]:
areas = np.arange(100, 501, 5)
mono = [pc.die_cost(a, 18500, 0.10) for a in areas]
tiles = [pc.chiplet_cost([(a/2, 18500, 0.10), (a/2, 18500, 0.10)], 10, 0.97) for a in areas]
first = next(a for a in range(100, 600) if pc.chiplet_cost([(a/2, 18500, 0.1), (a/2, 18500, 0.1)], 10, 0.97) < pc.die_cost(a, 18500, 0.1))
print(f'tiles cheaper from {first} mm^2')
fig, ax = plt.subplots(figsize=(6, 3.5)); ax.plot(areas, mono, label='monolithic'); ax.plot(areas, tiles, label='two tiles')
ax.set_xlabel('total area (mm^2)'); ax.set_ylabel('cost per good part ($)'); ax.legend(); ax.grid(True, alpha=0.3); fig.tight_layout()

tiles cheaper from 225 mm^2


## Discussion

Compare your predictions with the [lab README](README.md). Which of the four effects would change your choice between two laptops with the same chip?